In [ ]:
import os
import json
from bs4 import BeautifulSoup
from nltk.tokenize import sent_tokenize
import nltk

nltk.download('punkt')  # Download tokenizer model

RAW_DATA_FOLDER_PATH = "/home/madhavbpanicker/Documents/Scrape_project/Website-Data-Raw/"

# Function to clean HTML and extract sentences
def extract_relevant_sentences(html_content, search_terms):
    if not html_content:  # Check for None or empty content
        return []
    try:
        # Parse HTML
        soup = BeautifulSoup(html_content, 'html.parser')
        text = soup.get_text(separator=" ")  # Get clean text from HTML
        
        # Tokenize into sentences
        sentences = sent_tokenize(text)
        
        # Filter relevant sentences
        relevant_sentences = [
            sentence for sentence in sentences
            if any(term in sentence.lower() for term in search_terms)
        ]
        
        return relevant_sentences
    except Exception as e:
        print(f"Error extracting sentences: {e}")
        return []

# Process JSON files
def process_json_files(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    
    for file_name in os.listdir(input_folder):
        if file_name.endswith('.json'):
            input_file_path = os.path.join(input_folder, file_name)
            print(f"Processing file: {input_file_path}")
            
            # Extract search terms from the file name
            base_name = os.path.splitext(file_name)[0]
            search_terms = base_name.split('-')  # Assuming hyphens separate terms
            search_terms = [term.lower() for term in search_terms]  # Convert to lowercase
            
            with open(input_file_path, 'r', encoding='utf-8') as file:
                data = json.load(file)
            
            # Process each entry in the JSON file
            for entry in data:
                html_content = entry.get('page_content', '')  # Default to empty string if missing
                relevant_sentences = extract_relevant_sentences(html_content, search_terms)
                entry['relevant_sentences'] = relevant_sentences
            
            # Save updated results to a new JSON file
            output_file_path = os.path.join(output_folder, f"processed-{file_name}")
            with open(output_file_path, 'w', encoding='utf-8') as output_file:
                json.dump(data, output_file, ensure_ascii=False, indent=4)
            
            print(f"Processed results saved to {output_file_path}")

# Paths
input_folder = RAW_DATA_FOLDER_PATH  # Replace with your input folder path
output_folder = "/home/madhavbpanicker/Documents/Scrape_project/Website-Data-Trimmed/"  # Replace with your desired output folder

# Process files
process_json_files(input_folder, output_folder)
